In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import os
from adjustText import adjust_text
from vpei.common_utils import trim_model_names
from vpei.epistemic_consistency.results_utils import (
    compute_models_overall_bias_in_person_attribution_experiments,
    compute_models_overall_bias_in_politicized_context_experiments,
)
from vpei.models import MODELS, MODELS_WITH_REASON_OFF

experimental_results_path = '~/repos/epistemic_consistency_paper/experimental_results'
models = MODELS_WITH_REASON_OFF

# --- Person attribution bias ---
experiments_person = {
    "absolute_experiment": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions","cvs"],
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_with_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions","cvs"],
    "comparative_experiment_without_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions","cvs"],
}
df_person = compute_models_overall_bias_in_person_attribution_experiments(
    models=models,
    experiments_types_and_names_to_load=experiments_person,
    target_statistic='log_odds',
    experimental_results_path=experimental_results_path,
)
df_person.rename(columns={"MODEL MEAN": "model_mean"}, inplace=True)

# --- Politicized context bias ---
evaluate_experiments = [
    "evaluate_time_series_trends",
    "evaluate_research_designs",
    "evaluate_governments_based_on_country_metrics",
    "evaluate_factuality_of_news_articles",
    "evaluate_policy_proposals",
    "evaluate_two_group_comparison_policy_effectiveness",
    "evaluate_correlation_btw_governments_and_problem_metrics",
    "evaluate_protesters_behavior",
    "evaluate_social_media_posts",
    "evaluate_policy_effectiveness_given_contingency_tables",
]
df_politicized = compute_models_overall_bias_in_politicized_context_experiments(
    models=models,
    experiments_types_and_names_to_load={"unblind_experiment": evaluate_experiments},
    target_statistic='log_odds',
    experimental_results_path=experimental_results_path,
)
df_politicized.rename(columns={"MODEL MEAN": "model_mean"}, inplace=True)


In [ ]:

# --- Helper for a single scatter subplot ---
def get_model_display_label(row):
    long_name = MODELS.get(row['model_name'], {}).get('long_name', row.get('long_name', row['model_name']))
    label = long_name.split(' ', 1)[-1] if ' ' in long_name else long_name
    return trim_model_names([label])[0]


def scatter_subplot(ax, merged, x_col, x_label, y_label, title, expand=(0.2, 0.4), force_text=(0.2, 0.4)):
    ax.scatter(merged[x_col], merged['model_mean'], s=80, color='C2', zorder=3)
    texts = []
    for _, row in merged.iterrows():
        label = get_model_display_label(row)
        texts.append(ax.text(row[x_col], row['model_mean'], label, fontsize=10))
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5),
                expand=expand,
                force_text=force_text,)
    slope, intercept, r_value, p_value, _ = stats.linregress(merged[x_col], merged['model_mean'])
    x_range = np.linspace(merged[x_col].min(), merged[x_col].max(), 200)
    ax.plot(x_range, slope * x_range + intercept, color='firebrick', linewidth=3, linestyle='--',
            label=f'r = {r_value:.2f}, p = {p_value:.3f}')
    ax.axhline(0, color='gray', linewidth=3, linestyle='--')
    ax.set_xlabel(x_label, fontsize=14)
    ax.set_ylabel(y_label, fontsize=14)
    ax.set_title(title, fontsize=20, fontweight='bold')
    ax.legend(fontsize=15, loc='lower right')
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-1.2, 0.2)
    #set text on lower left corner with gray color and fontsize 10
    ax.text(0.02, 0.02, f'n = {len(merged)}', transform=ax.transAxes, color='gray', fontsize=15)

# --- Load LM Arena ratings and merge ---
arena_df = pd.read_csv(os.path.expanduser(
    '~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_llm_arena_rating.csv'
))
arena_df = arena_df.dropna(subset=['arena_rating'])

merged_person_arena = arena_df.merge(df_person, on='model_name', how='inner')
merged_politicized_arena = arena_df.merge(df_politicized, on='model_name', how='inner')

In [ ]:
# --- Person attribution panel: bias vs LM Arena rating ---
fig, ax = plt.subplots(figsize=(8, 7))
scatter_subplot(
    ax, merged_person_arena,
    x_col='arena_rating',
    x_label='LM Text Arena Rating',
    y_label='Political Bias — Model Mean Log Odds',
    title='Person Attribution Experiments\nvs. LM Text Arena Rating',
    expand=(2, 2),
    force_text=(2, 2),    
)
plt.tight_layout()
fig.savefig('./figures/scatterplot_person_attribution_vs_lm_arena.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Politicized context panel: bias vs LM Arena rating ---
fig, ax = plt.subplots(figsize=(8, 7))
scatter_subplot(
    ax, merged_politicized_arena,
    x_col='arena_rating',
    x_label='LM Text Arena Rating',
    y_label='Political Bias — Model Mean Log Odds',
    title='Politicized Context Experiments\nvs. LM Text Arena Rating',
    expand=(1, 1),
    force_text=(1, 1),
)
plt.tight_layout()
fig.savefig('./figures/scatterplot_politicized_context_vs_lm_arena.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Load ECI scores and merge ---
eci_df = pd.read_csv(os.path.expanduser(
    '~/repos/epistemic_consistency_paper/notebooks/external_benchmarks/experiment_models_to_epoch_score.csv'
))
eci_df = eci_df.dropna(subset=['eci'])

merged_person_eci = eci_df.merge(df_person, on='model_name', how='inner')
merged_politicized_eci = eci_df.merge(df_politicized, on='model_name', how='inner')

# --- Person attribution panel: bias vs ECI score ---
fig, ax = plt.subplots(figsize=(8, 7))
scatter_subplot(
    ax, merged_person_eci,
    x_col='eci',
    x_label='ECI Score (Epoch)',
    y_label='Political Bias — Model Mean Log Odds',
    title='Person Attribution Experiments\nvs. ECI Score',
    expand=(2, 2),
    force_text=(2, 2),        
)
plt.tight_layout()
fig.savefig('./figures/appendix_scatterplot_person_attribution_vs_eci.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Politicized context panel: bias vs ECI score ---
fig, ax = plt.subplots(figsize=(8, 7))
scatter_subplot(
    ax, merged_politicized_eci,
    x_col='eci',
    x_label='ECI Score (Epoch)',
    y_label='Political Bias — Model Mean Log Odds',
    title='Politicized Context Experiments\nvs. ECI Score',
    expand=(1, 1),
    force_text=(1, 1),        
)
plt.tight_layout()
fig.savefig('./figures/appendix_scatterplot_politicized_context_vs_eci.png', dpi=300, bbox_inches='tight')
plt.show()